In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col, sum as spark_sum, count

In [0]:
# Step 2: read the CSV file into a Spark DataFrame

file_path = "/Volumes/workspace/default/sales_data/sales.csv"
sales_df = (
    spark.read
        .format("csv")
        .option("header", "true")        # first row holds the column names
        .option("inferSchema", "true")   # let Spark work out the data types
        .load(file_path)
)

sales_df.printSchema()
display(sales_df)

root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- sale: integer (nullable = true)
 |-- city: string (nullable = true)



id,date,sale,city
100,2023-10-28,4002,Suva
101,2023-08-29,5983,Alor Star
102,2023-10-15,6643,Adak
103,2023-10-03,5027,Palikir
104,2023-10-19,3667,Mexicali
105,2023-10-25,5517,Alofi
106,2023-09-01,8122,Anadyr (town)
107,2023-10-04,5903,Kaliningrad
108,2023-09-13,8063,Lahore
109,2023-10-02,6885,Rome


In [0]:
table_name = "workspace.default.sales"

(
    sales_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
)

print("Table created:", table_name)

Table created: workspace.default.sales


In [0]:
sales = spark.table("workspace.default.sales")

print("Row count:", sales.count())
display(sales.limit(5))

Row count: 1000


id,date,sale,city
100,2023-10-28,4002,Suva
101,2023-08-29,5983,Alor Star
102,2023-10-15,6643,Adak
103,2023-10-03,5027,Palikir
104,2023-10-19,3667,Mexicali


In [0]:
# clear any widgets 
dbutils.widgets.removeAll()

# the dropdown for user filtering
dbutils.widgets.dropdown(
    name="frequency",                          
    defaultValue="Daily",                      
    choices=["Daily", "Weekly", "Monthly"],    
    label="Report Frequency"                   
)

# a run date, so we can pretend "today" is any date we like
dbutils.widgets.text(
    name="run_date",
    defaultValue="2023-11-11",
    label="Run Date (yyyy-MM-dd)"
)

In [0]:
# Step 4b: read the widget values into Python variables

frequency = dbutils.widgets.get("frequency")
run_date_str = dbutils.widgets.get("run_date")

print("Frequency selected:", frequency)
print("Run date entered:", run_date_str)

Frequency selected: Monthly
Run date entered: 2023-10-10


In [0]:
# Step 5: work out the date window from the widget values
frequency = dbutils.widgets.get("frequency")
run_date_str = dbutils.widgets.get("run_date")

# widgets always give text, so convert it into a real date
run_date = datetime.strptime(run_date_str, "%Y-%m-%d").date()

# the report always ends yesterday, because today is still incomplete
end_date = run_date - timedelta(days=1)

if frequency == "Daily":
    start_date = end_date

elif frequency == "Weekly":
    start_date = end_date - timedelta(days=6)   # 6 days back plus yesterday = 7 days

elif frequency == "Monthly":
    start_date = end_date.replace(day=1)        # 1st of yesterday's month

else:
    raise ValueError(f"Unknown frequency: {frequency}")

print("Frequency :", frequency)
print("Start date:", start_date)
print("End date  :", end_date)

Frequency : Monthly
Start date: 2023-10-01
End date  : 2023-10-09


In [0]:
# Step 6: filter the table to the date window and aggregate

from pyspark.sql.functions import col, sum as spark_sum, count

sales = spark.table("workspace.default.sales")

report_df = (
    sales
        .filter((col("date") >= start_date) & (col("date") <= end_date))
        .groupBy("city")
        .agg(
            spark_sum("sale").alias("total_sales"),
            count("*").alias("number_of_orders")
        )
        .orderBy(col("total_sales").desc())
)

print(f"Report for {frequency}: {start_date} to {end_date}")
print("Cities in report:", report_df.count())

Report for Weekly: 2023-10-03 to 2023-10-09
Cities in report: 88


In [0]:
display(report_df)

city,total_sales,number_of_orders
Belize City,12084,2
Nuuk,9843,1
Chengdu,9789,1
Kanpur,9713,1
Belfast,9618,1
Xining,9608,1
Porto Alegre,9523,1
Assis,9212,1
Innsbruck,9198,1
Guangzhou,9128,1


In [0]:
summary = (
    sales
        .filter((col("date") >= start_date) & (col("date") <= end_date))
        .agg(
            spark_sum("sale").alias("total_sales"),
            count("*").alias("total_orders")
        )
)

display(summary)

total_sales,total_orders
516457,93


In [0]:
output_path = (
    f"/Volumes/workspace/default/sales_data/reports/"
    f"{frequency.lower()}/{end_date}"
)

(
    report_df
        .coalesce(1)                  # force a single output file instead of many
        .write
        .mode("overwrite")            # re-running the same report replaces it
        .option("header", "true")     # include the column names row
        .csv(output_path)
)

print("Report written to:", output_path)

Report written to: /Volumes/workspace/default/sales_data/reports/weekly/2023-10-09


In [0]:
# Step 8b: confirm what was actually written
display(dbutils.fs.ls(output_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/sales_data/reports/daily/2023-11-10/_SUCCESS,_SUCCESS,0,1789465468000
dbfs:/Volumes/workspace/default/sales_data/reports/daily/2023-11-10/_committed_4726175578174226546,_committed_4726175578174226546,113,1789388791000
dbfs:/Volumes/workspace/default/sales_data/reports/daily/2023-11-10/_committed_6270998140508196551,_committed_6270998140508196551,212,1789465468000
dbfs:/Volumes/workspace/default/sales_data/reports/daily/2023-11-10/_committed_vacuum4348231954621803292,_committed_vacuum4348231954621803292,96,1789465468000
dbfs:/Volumes/workspace/default/sales_data/reports/daily/2023-11-10/_started_6270998140508196551,_started_6270998140508196551,0,1789465467000
dbfs:/Volumes/workspace/default/sales_data/reports/daily/2023-11-10/part-00000-tid-6270998140508196551-78f99ca9-133a-4aa5-9396-6936d6ddaedc-142-1-c000.csv,part-00000-tid-6270998140508196551-78f99ca9-133a-4aa5-9396-6936d6ddaedc-142-1-c000.csv,178,1789465467000


In [0]:
# Step 8c: copy the part file out under a readable name

files = dbutils.fs.ls(output_path)
part_file = [f.path for f in files if f.name.startswith("part-")][0]

final_file = (
    f"/Volumes/workspace/default/sales_data/reports/"
    f"sales_report_{frequency.lower()}_{end_date}.csv"
)

dbutils.fs.cp(part_file, final_file)
print("Clean file:", final_file)

Clean file: /Volumes/workspace/default/sales_data/reports/sales_report_daily_2023-11-10.csv


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/sales_data/reports"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/sales_data/reports/daily/,daily/,0,1789466080728
dbfs:/Volumes/workspace/default/sales_data/reports/sales_report_daily_2023-11-10.csv,sales_report_daily_2023-11-10.csv,178,1789466067000
